In [ ]:
import copy
import getopt
import math
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import operator
import os
import random
import sys
from scipy import stats
import time
import bezier
import numpy as np
%matplotlib inline  

In [ ]:
def clean(series):
    series = series.str.upper()
    series = series.str.replace(u"Á", "A")
    series = series.str.replace(u"É", "E")
    series = series.str.replace(u"Ú", "U")
    series = series.str.replace(u"Ó", "O")
    series = series.str.replace(u"Í", "I")
    series = series.str.replace(u"Ã", "A")
    series = series.str.replace(u"Õ", "O")
    series = series.str.replace(u"Ô", "O")
    series = series.str.replace(u"Â", "A")
    series = series.str.replace(u"Ê", "E")
    series = series.str.replace(u"Ç", "C")
    series = series.str.replace(u"-", "")
    series = series.str.replace(u"'", "")
    series = series.str.upper()
    return series

def clean_str(series):
    series = series.upper()
    series = series.replace(u"Á", "A")
    series = series.replace(u"É", "E")
    series = series.replace(u"Ú", "U")
    series = series.replace(u"Ó", "O")
    series = series.replace(u"Í", "I")
    series = series.replace(u"Ã", "A")
    series = series.replace(u"Õ", "O")
    series = series.replace(u"Ô", "O")
    series = series.replace(u"Â", "A")
    series = series.replace(u"Ê", "E")
    series = series.replace(u"Ç", "C")
    series = series.replace(u"-", "")
    series = series.replace(u"'", "")
    return series

In [ ]:
global VISUALIZE

In [ ]:
def main():
    """
    Primary function that initiates network creation and handles execution of
    infection simulations.
    Args:
        argv: A list of command-line arguments passed to the application.
    Returns:
        Void
    """

    # Flag defaults
    VISUALIZE = False
    DELAY = 0
    NUM_SIMULATIONS = 1
    
    # Determine the parameters of the current simulationn
    #opts, args = getopt.getopt(sys.argv[1:], "brcsidv", ["delay=10","nsim=2"])
    
    opts = [('-v', ''),('-b', '')]

    simulations = list()

    for o, a in opts:
        if o == "-b":
            simulations.append("betweenness")
        elif o == "-r":
            simulations.append("random")
        elif o == "-s":
            simulations.append("sir")
        elif o == "-c":
            simulations.append("clustering")
        elif o == "-v":
            VISUALIZE = True
        elif o == "-y":
            RECALCULATE = False
        elif o == "--delay":
            DELAY = int(a)
        elif o == "--nsim":
            NUM_SIMULATIONS = int(a)
        

    #seed = 100
    #random.seed(seed)

    # Identify the script.
    
    print('VISUALIZE =', VISUALIZE)
    print('DELAY =', DELAY)
    print('NUM_SIMULATIONS =', NUM_SIMULATIONS,"\n\n")
    print('SIMULATIONS =', simulations,"\n\n")

    # Create the network using the command arguments.
    network = create_network('grafo_cidades_aeroportos.gexf')
    edgepool = network.edges(data=True)
  
    # Generate target-selection weights, and choose target vertices to infect.
    weights = nx.get_node_attributes(network, 'population')
    #degrees = network.degree()
    #weights = dict()
    #for city, degree in degrees:
    #    weights[city] = network.degree(city)
    targets = list()
    
    for ind in range(0,NUM_SIMULATIONS):
        target_round = list()
        target_round.append('SAO PAULO (SP)')
        while len(target_round) < 3:
             chosen_city = weighted_random(weights)
             if chosen_city not in target_round:
                 target_round.append(chosen_city)
        targets.append(target_round)
        
    print('TARGETS =',targets, "\n")

    # Make a directory for the data, and change into that directory.
    currenttime = time.strftime("%Y-%m-%dT%H%M%S", time.gmtime())
    os.makedirs(currenttime)
    os.chdir(currenttime)

    # Record relevent data about the simulation.
    # TMP simulation_data(network, currenttime, target, seed)

    # Prepare simulations

    for strategy in simulations:
    
        print("{0} Mode.".format(strategy) )

        # Generate a list a sorted list of flights to cancel based on the
        # strategy.
        
        cancellist = list()
        if strategy == "random":
            # Sort the edges randomly
            cancellist = random.sample(edgepool, len(edgepool))

        elif strategy == "clustering":
            # Sort the edges based on the sum of the clustering coefficent.
            sorted_cluster = sorted(edgepool, key=lambda k: k[2]['cluster'],
                            reverse=True)
            for cluster_item in sorted_cluster:
                if network[cluster_item[0]][cluster_item[1]]['cluster'] < 2:
                    if network[cluster_item[0]][cluster_item[1]]['cluster'] > 0:
                        cancellist.append((cluster_item[0], cluster_item[1]))

        elif strategy == "betweenness":
            # Sort the edges based on weighted edge-betweenness.
            betweennesses = nx.edge_betweenness_centrality(network,
                                                           weight="weight")
            cancellist = sorted(betweennesses.keys(), 
                                key=lambda k: betweennesses[k], reverse=True)


        print(cancellist[:20])
        # Make a new folder for the data.
        os.makedirs(strategy)
        
        iteration = 0
        efforts = [0]
        #efforts.extend(range(1,101,5))
        for target in targets:

            # Open a file for this targets dataset
            output_file = open("{0}/{0}_{1}.csv".format(strategy,
                                                        pad_string(iteration,4)
                                                        ),"w")
            output_file.write('"effort","total_infected, edges_closed"\n')


            for effort in efforts:
                if effort != 0:
                    max_index = int(len(cancellist) * (effort/100))-1
                    cancelled = cancellist[0:max_index]
                else:
                    cancelled = None

                title = "{0} - {1}%".format(strategy, effort/100)
                print(target)
                results = infection(network, cancelled, target, vis=VISUALIZE,
                                    title=title, DELAY=DELAY)
                total_infected = results["Infected"] + results["Recovered"]
                output_file.write("{0},{1}\n".format(effort/100,total_infected))
                
                if total_infected == 1:
                    for remaining_effort in range(effort+5,101,5):
                        output_file.write("{0},{1}\n".format(remaining_effort/100,
                                                              total_infected))
                    break

            iteration += 1
            output_file.close()



In [ ]:
def weighted_random(weights):
    number = random.random() * sum(weights.values())
    for k,v in weights.items():
        if number <= v:
            break
        number -= v
    return k

In [ ]:
def pad_string(integer, n):
    """
    Add "0" to the front of an interger so that the resulting string in n 
    characters long.
    Args:
        integer: The number to pad.
        n: The desired length of the string
    Returns
        string: The padded string representation of the integer.
        
    """

    string = str(integer)

    while len(string) < n:
        string = "0" + string

    return string

In [ ]:
def curved_edges(G, pos, dist_ratio=0.2, bezier_precision=20, polarity='random'):
    # Get nodes into np array
    edges = np.array(G.edges())
    l = edges.shape[0]

    if polarity == 'random':
        # Random polarity of curve
        rnd = np.where(np.random.randint(2, size=l)==0, -1, 1)
    else:
        # Create a fixed (hashed) polarity column in the case we use fixed polarity
        # This is useful, e.g., for animations
        rnd = np.where(np.mod(np.vectorize(hash)(edges[:,0])+np.vectorize(hash)(edges[:,1]),2)==0,-1,1)
    
    # Coordinates (x,y) of both nodes for each edge
    # e.g., https://stackoverflow.com/questions/16992713/translate-every-element-in-numpy-array-according-to-key
    # Note the np.vectorize method doesn't work for all node position dictionaries for some reason
    u, inv = np.unique(edges, return_inverse = True)
    coords = np.array([pos[x] for x in u])[inv].reshape([edges.shape[0], 2, edges.shape[1]])
    coords_node1 = coords[:,0,:]
    coords_node2 = coords[:,1,:]
    
    # Swap node1/node2 allocations to make sure the directionality works correctly
    should_swap = coords_node1[:,0] > coords_node2[:,0]
    coords_node1[should_swap], coords_node2[should_swap] = coords_node2[should_swap], coords_node1[should_swap]
    
    # Distance for control points
    dist = dist_ratio * np.sqrt(np.sum((coords_node1-coords_node2)**2, axis=1))

    # Gradients of line connecting node & perpendicular
    m1 = (coords_node2[:,1]-coords_node1[:,1])/(coords_node2[:,0]-coords_node1[:,0])
    m2 = -1/m1

    # Temporary points along the line which connects two nodes
    # e.g., https://math.stackexchange.com/questions/656500/given-a-point-slope-and-a-distance-along-that-slope-easily-find-a-second-p
    t1 = dist/np.sqrt(1+m1**2)
    v1 = np.array([np.ones(l),m1])
    coords_node1_displace = coords_node1 + (v1*t1).T
    coords_node2_displace = coords_node2 - (v1*t1).T

    # Control points, same distance but along perpendicular line
    # rnd gives the 'polarity' to determine which side of the line the curve should arc
    t2 = dist/np.sqrt(1+m2**2)
    v2 = np.array([np.ones(len(edges)),m2])
    coords_node1_ctrl = coords_node1_displace + (rnd*v2*t2).T
    coords_node2_ctrl = coords_node2_displace + (rnd*v2*t2).T

    # Combine all these four (x,y) columns into a 'node matrix'
    node_matrix = np.array([coords_node1, coords_node1_ctrl, coords_node2_ctrl, coords_node2])

    # Create the Bezier curves and store them in a list
    curveplots = []
    for i in range(l):
        nodes = node_matrix[:,i,:].T
        curveplots.append(bezier.Curve(nodes, degree=2).evaluate_multi(np.linspace(0,1,bezier_precision)).T)
      
    # Return an array of these curves
    curves = np.array(curveplots)
    return curves

In [ ]:
def create_network(path):
    """
    Create a NetworkX graph object using the airport and route databases.
    Args:
        nodes: The file path to the nodes .csv file.
        edeges: The file path to the edges .csv file.
    Returns:
        G: A NetworkX DiGraph object populated with the nodes and edges assigned
           by the data files from the arguments.
           
   
    """
    print("Creating network.")
    G = nx.read_gexf(path)
    
    # Add pos attribute in nodes
    for node in G.nodes():
        long = G.nodes[node]['long']
        lat = G.nodes[node]['lat']
        loc = (long, lat)
        node_dict = {node : loc}
        nx.set_node_attributes(G, node_dict, 'pos')
    
    # Calculate the edge weights
    print("\tCalculating edge weights",end="")
    degree_network = nx.Graph(G)
    ldegree = degree_network.degree
    for i,j in G.edges():
        degree_sum = ldegree[i] + ldegree[j]
        G[i][j]['weight'] = degree_sum
    
    print("\t\t\t\t[Done]")
    
    G = nx.relabel_nodes(G, clean_str)
    
    # Calculate the edge distances
    print("\tCalculating edge distance",end="")
    G = calculate_distance(G)
    print("\t\t\t\t[Done]")

    
    # Add clustering data
    print("\tCalculating clustering coefficents",end="")
    cluster_network = nx.Graph(G)
    lcluster = nx.clustering(cluster_network)
    for i,j in G.edges():
        cluster_sum = lcluster[i] + lcluster[j]
        G[i][j]['cluster'] = cluster_sum
    print("\t\t\t[Done]")
    
    return G

In [ ]:
from math import cos, asin, sqrt, pi

def distance(lat1, lon1, lat2, lon2):
    p = pi/180
    a = 0.5 - cos((lat2-lat1)*p)/2 + cos(lat1*p) * cos(lat2*p) * (1-cos((lon2-lon1)*p))/2
    return 12742 * asin(sqrt(a))

def calculate_distance(input_network):
    """
    Add weights to the edges of a network based on the degrees of the connecting
    verticies, and return the network.
    Args:
        input_network: A NetworkX graph object
    Returns:
        G: A weighted NetworkX graph object.
    """
    
    G = input_network.copy()

    # Add weights to edges
    for node, successor in G.edges():
        dist = distance(G.nodes[node]['lat'], G.nodes[node]['long'],
                        G.nodes[successor]['lat'], G.nodes[successor]['long'])
        edge_dict = {(node,successor): dist}
        nx.set_edge_attributes(G, edge_dict, 'distance')
    return G

In [ ]:
def calculate_weights(input_network):
    """
    Add weights to the edges of a network based on the degrees of the connecting
    verticies, and return the network.
    Args:
        input_network: A NetworkX graph object
    Returns:
        G: A weighted NetworkX graph object.
        
    G = input_network.copy()

    # Add weights to edges
    for node in G.nodes():
        successors = G.neighbors(node)
        weights = {}

        # Calculate the total out degree of all succs
        total_degree = 0
        for successor in successors: 
            weights[successor] = G.degree(successor)
            
        largest_weight = max(weights.items())[1]
        smallest_weight = min(weights.items())[1]
        
        successors = G.neighbors(node)
        for successor in successors:
            if largest_weight == smallest_weight:
                relative_weight = 0
            else: 
                relative_weight = (weights[successor] - smallest_weight) / (largest_weight - smallest_weight)    
            G[node][successor]['weight'] = relative_weight
    """
    
    G = input_network.copy()

    # Add weights to edges
    for node in G.nodes():
        successors = G.neighbors(node)
        weights = {}

        # Calculate the total out degree of all succs
        total_degree = 0
        for successor in successors: 
            weights[successor] = G.degree(successor)
            
        largest_weight = max(weights.items())[1]
        smallest_weight = min(weights.items())[1]
        
        successors = G.neighbors(node)
        for successor in successors:
            if largest_weight == smallest_weight:
                relative_weight = 0
            else: 
                relative_weight = (weights[successor] - smallest_weight) / (largest_weight - smallest_weight)    
            G[node][successor]['weight'] = relative_weight

    return G


In [ ]:
def visualize_real(network, title,pos, curves):
    """
    Visualize the network given an array of posisitons.
    """
    print("-- Starting to Visualize --")

    colors = []
    alphas = []
    edge_colors = []
    for node in network.nodes():
        colors.append(network.nodes[node]["color"])
        alphas.append(network.nodes[node]["alpha"])
    for edge in network.edges(data=True):
        edge_colors.append(network[edge[0]][edge[1]]["color"])
   
    positions = nx.get_node_attributes(network, 'pos')
    lc = LineCollection(curves, color=edge_colors, alpha=0.1)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10,10))
    nx.draw_networkx_nodes(network, positions, node_size=15, alpha=alphas, node_color = colors, ax=ax)
    plt.gca().add_collection(lc)
    plt.tick_params(axis='both',which='both',bottom=False,left=False,labelbottom=False,labelleft=False)
    plt.show()

    number_files = str(len(os.listdir()))
    while len(number_files) < 3:
        number_files = "0" + number_files

    #plt.savefig("infection-{0}.png".format(number_files),
    #            bbox_inches='tight', dpi=600 
    #           )
    #plt.show()
    #plt.close()

In [ ]:
def infection(input_network, vaccination, starts,DELAY=0, vis = False, 
              file_name = "sir.csv", title="",  RECALCULATE = False):
    """
    Simulate an infection within network, generated using seed, and with the
    givin vaccination strategy. This function will write data from each timestep
    to "data.csv".
    Args:
        network: A NetworkX DiGraph object.
        vaccination: A list of node indices to label as recovered from the 
                     begining.
    Returns:
        state: A dictionary of the total suscceptable, infected, and recovered.
    """

    print("Simulating infection.")

    network = input_network.copy()
    positions = nx.get_node_attributes(network , 'pos')
    curves = curved_edges(network , positions)
    
    # Recalculate the weights of the network as per necessary

    # Open the data file
    f = open(file_name, "w")
    f.write("time, s, e, i, r\n")

    # Set the default to susceptable
    sys.stdout.flush()
    for node in network.nodes():
        nx.set_node_attributes(network, values = {node:"s"}, name='status')
        nx.set_node_attributes(network, values = {node:(0.48942421, 0.72854938, 0.56751036)}, name='color')
        nx.set_node_attributes(network, values = {node:0}, name='age')
        nx.set_node_attributes(network, values = {node: 0.4}, name='alpha')
    for edge in network.edges(data=True):
        nx.set_edge_attributes(network, {(edge[0],edge[1]): "#A6A6A6"}, 'color')
    
    #S is the fraction of susceptible individuals (those able to contract the disease), (0.48942421, 0.72854938, 0.56751036)
    #E is the fraction of exposed individuals (those who have been infected but are not yet infectious),(0.9155979, 0.55210684, 0.42070204)
    #I is the fraction of infective individuals (those capable of transmitting the disease),(0.81942908, 0.28911553, 0.38102921)
    #R is the fraction of recovered individuals (those who have become immune). (0.42355299, 0.16934709, 0.42581586)
    
    # Assign the infected
    for start in starts:
        infected = start
        network.nodes[infected]["status"] = "i"
        network.nodes[infected]["color"]  = (0.81942908, 0.28911553, 0.38102921)
        
    if vaccination is not None:
        print("\tVaccinated: ", len(vaccination) )
    else: 
        print("\tVaccinated: None")

    if vis:
        pos = nx.get_node_attributes(network, 'pos')
    
    decay = 1

    # Iterate through the evolution of the disease.
    for step in range(0,99):
        # If the delay is over, vaccinate.
        # Convert the STRING! 
        if int(step) == int(DELAY):
            if vaccination is not None:
                print(DELAY,"on step",step)
                network.remove_edges_from(vaccination)
                # Recalculate the weights of the network as per necessary
                if RECALCULATE == True:
                    network = calculate_weights(network)


        # Create variables to hold the outcomes as they happen
        S,E,I,R = 0,0,0,0

        for node in network.nodes():
            status = network.nodes[node]["status"]
            age = network.nodes[node]["age"]
            color = network.nodes[node]["color"]

            if status == "i" and age >= 4:
                # The infected has reached its recovery time
                #network.nodes[node]["status"] = "r"
                #network.nodes[node]["color"] = (0.42355299, 0.16934709, 0.42581586)
                network.nodes[node]["status"] = "s"
                network.nodes[node]["color"] = (0.48942421, 0.72854938, 0.56751036)
                
            if status == "e" and age >= 2 and age < 4:
                # The infected has reached its recovery time
                network.nodes[node]["status"] = "i"
                network.nodes[node]["color"] = (0.81942908, 0.28911553, 0.38102921)

            elif status == "e":
                network.nodes[node]["age"] += 1

            elif status == "i":
                # Propogate the infection.
                if age > 0:
                    victims = network.neighbors(node)
                    number_infections = 0
                    for victim in victims:
                        infect_status = network.nodes[victim]["status"]
                        infect = False # Set this flag to False to start 
                                       # weighting.
                        if random.uniform(0,1) <= decay:#network[node][victim]['weight']:
                            infect = True
                            number_infections+=1

                        if infect_status == "s" and infect == True:
                            network.nodes[victim]["status"] = "e"
                            network.nodes[victim]["age"] = 0
                            network.nodes[victim]["color"] = (0.9155979, 0.55210684, 0.42070204)
                network.nodes[node]["age"] += 1

        decay = decay*0.97
        # Loop twice to prevent bias.
        for node in network.nodes():
            status = network.nodes[node]["status"]
            age = network.nodes[node]["age"]
            color = network.nodes[node]["color"]

            if status == "s":
                # Count those susceptable
                S += 1

            if status == "e":
                E += 1

            if status == "v":
                S += 1

            elif status == "r":

                R += 1

            elif status == "i":
                
                I += 1
        print("{0}, {1}, {2}, {3}, {4}".format(step, S, E, I, R))

        printline = "{0}, {1}, {2}, {3}, {4}".format(step, S, E, I, R)
        f.write(printline + "\n")

       # print("\t"+printline)

        if I == 0:
            break

        if vis:
            #write_dot(network, title+".dot")
            visualize_real(network, title, pos, curves)
        
    print("\t----------\n\tS: {0}, I: {1}, R: {2}".format(S,I,R))

    return {"Suscceptable":S,"Infected":I, "Recovered":R}

In [ ]:
def visualize(network, title,pos):
    """
    Visualize the network given an array of posisitons.
    """
    print("-- Starting to Visualize --")
    MAP = False

    if MAP:
        m = Basemap(
            projection='cea',
            llcrnrlat=-90, urcrnrlat=90,
            llcrnrlon=-180, urcrnrlon=180,
            resolution=None
            )

        pos = dict()

        for pos_node in network.nodes():
            # Normalize the lat and lon values
            x,y = m(float(network.nodes[pos_node]['lon']),
                    float(network.nodes[pos_node]['lat']))
        
            pos[pos_node] = [x,y]


    colors = []
    i_edge_colors = []
    d_edge_colors = []
    default = []
    infected = []
    for node in network.nodes():
        colors.append(network.nodes[node]["color"])
    for i,j in network.edges():
        color = network.nodes[i]["color"]
        alpha = 0.75
        if color == (0.42355299, 0.16934709, 0.42581586) or color == (0.48942421, 0.72854938, 0.56751036):
            color = "#A6A6A6"
            default.append((i,j))
            d_edge_colors.append(color)
        else:
            color = "#A6A6A6"#29A229"
            infected.append((i,j))
            i_edge_colors.append(color)

    plt.figure(figsize=(7,7))

    # Fist pass - Gray lines
    nx.draw_networkx_edges(network,pos,edgelist=default,
            width=0.5,
            edge_color=d_edge_colors,
            alpha=0.5,
            arrows=False)
   
    # Second Pass - Colored lines
    nx.draw_networkx_edges(network,pos,edgelist=infected,
            width=0.5,
            edge_color=i_edge_colors,
            alpha=0.75,
            arrows=False)
    
    size = nx.get_node_attributes(network, 'population')
    size_list = [v for v in size.values()]
    size_norm = [float(i)/max(size_list) for i in size_list]
    nx.draw_networkx_nodes(network,
            pos,
            linewidths=0.5,
            node_size=[s*7000 for  s in size_norm],
            alpha=0.75,               
            #with_labels=False,
            node_color = colors)
    
    # Adjust the plot limits
    cut = 1.05
    xmax = cut * max(xx for xx,yy in pos.values())
    xmin =  min(xx for xx,yy in pos.values())
    xmin = xmin - (cut * xmin)


    ymax = cut * max(yy for xx,yy in pos.values())
    ymin = (cut) * min(yy for xx,yy in pos.values())
    ymin = ymin - (cut * ymin)

    #plt.xlim(xmin,xmax)
    #plt.ylim(ymin,ymax)

    if MAP:
        # Draw the map
        m.bluemarble()
    plt.title=title

    plt.axis('off')

    number_files = str(len(os.listdir()))
    while len(number_files) < 3:
        number_files = "0" + number_files

    plt.savefig("infection-{0}.png".format(number_files),
                bbox_inches='tight', dpi=600 
            )
    plt.show()
    plt.close()

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
G = create_network('grafo_cidades_aeroportos.gexf')

In [ ]:
G = create_network('/Users/ana/grafo_cidades_aeroportos.gexf')
infection(G, [], ['SAO PAULO (SP)'], vis=True,
                                    title='title', DELAY=0)

In [ ]:
import glob
filenames = glob.glob("/Users/ana/2021-06-08T181611/*.png")

In [ ]:
import imageio
images = []
for filename in filenames:
    print(filename)
    images.append(imageio.imread(filename))
imageio.mimsave('/Users/ana/grafo2.gif', images)

In [ ]:
G = create_network('/Users/ana/grafo_cidades.gexf')
infection(G, [], ['São Paulo (SP)','Rio de Janeiro (RJ)'], vis=True,
                                    title='title', DELAY=0)

In [ ]:
colors = []
i_edge_colors = []
d_edge_colors = []
default = []
infected = []
for node in network.nodes():
    colors.append(network.nodes[node]["color"])
for i,j in network.edges():
    color = network.nodes[i]["color"]
    alpha = 0.75
    if color == "#A0C8F0" or color == "#FF6F00" or color == "purple":
        color = "#A6A6A6"
    default.append((i,j))
            d_edge_colors.append(color)
        else:
            color = "#29A229"
            infected.append((i,j))
            i_edge_colors.append(color)

    plt.figure(figsize=(7,7))

    # Fist pass - Gray lines
    nx.draw_networkx_edges(network,pos,edgelist=default,
            width=0.5,
            edge_color=d_edge_colors,
            alpha=0.5,
            arrows=False)
   
    # Second Pass - Colored lines
    nx.draw_networkx_edges(network,pos,edgelist=infected,
            width=0.5,
            edge_color=i_edge_colors,
            alpha=0.75,
            arrows=False)

    nx.draw_networkx_nodes(network,
            pos,
            linewidths=0.5,
            node_size=10,
            #with_labels=False,
            node_color = colors)
    
    # Adjust the plot limits
    cut = 1.05
    xmax = cut * max(xx for xx,yy in pos.values())
    xmin =  min(xx for xx,yy in pos.values())
    xmin = xmin - (cut * xmin)


    ymax = cut * max(yy for xx,yy in pos.values())
    ymin = (cut) * min(yy for xx,yy in pos.values())
    ymin = ymin - (cut * ymin)

    #plt.xlim(xmin,xmax)
    #plt.ylim(ymin,ymax)

    if MAP:
        # Draw the map
        m.bluemarble()
    plt.title=title

    plt.axis('off')

    number_files = str(len(os.listdir()))
    while len(number_files) < 3:
        number_files = "0" + number_files

    plt.savefig("infection-{0}.png".format(number_files),
                bbox_inches='tight', dpi=600 
            )
    plt.show()
    plt.close()